# ZelusBench: Attention Shifting (Short) — Transform Density 0.0-0.1

Can the model handle occasional transforms that invalidate prior computation?

**Independent variable**: transform density ∈ [0.0, 0.1]
- At 0.0, pure chain-following. At 0.1, ~10% of steps are rotations/reflections/translations
- Transforms cascade: rotating point D invalidates everything downstream
- depth=6, num_points=12, dim=3 fixed
- POSITION queries placed after transforms, query_min_depth=4
- 10 seeds per level = 20 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTH = 6
NUM_POINTS = 12
TRANSFORM_PROBS = [0.0, 0.1]
SEEDS = 10
TOTAL = len(TRANSFORM_PROBS) * SEEDS
print(f"ZelusBench Attention Shifting (Short) — Transform Density 0.0-0.1")
print(f"Depth: {DEPTH} | Points: {NUM_POINTS} | Transform probs: {TRANSFORM_PROBS}")
print(f"Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
def _prompt_with_retry(llm, prompt, attempts=4):
    for attempt in range(attempts):
        try:
            return llm.prompt(prompt)
        except Exception as e:
            if attempt == attempts - 1:
                raise
            wait = 2 ** attempt
            print(f"    !! API error ({type(e).__name__}), retry {attempt+1}/{attempts-1} in {wait}s")
            time.sleep(wait)

@kbench.task(name="zelusbench_shifting_short")
def zelusbench_shifting_short(llm) -> float:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scenario_avgs = []
    level_scores = {}
    scenario_num = 0
    for tp in TRANSFORM_PROBS:
        level_scores[tp] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "dim": 3,
                "min_chain_depth": DEPTH, "max_chain_depth": DEPTH,
                "num_points": NUM_POINTS,
                "transform_prob": tp,
                "query_types": [QueryType.POSITION],
                "query_min_depth": 4,
                "num_queries": 3,
                "seed": int(tp * 1000) + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"shifting_short_{tp}_{i}")
            response = _prompt_with_retry(llm, scenario.prompt)
            scores = score_response(response, scenario)
            avg = float(np.mean([s.score for s in scores]))
            level_scores[tp].append(avg)
            all_scenario_avgs.append(avg)
            print(f"  [{scenario_num}/{TOTAL}] tp={tp} avg={avg:.2f} | lb={cfg.leaf_bias}")
        print(f"  >> Transform prob {tp} mean: {float(np.mean(level_scores[tp])):.3f}")
    print("\n" + "=" * 60)
    for tp in TRANSFORM_PROBS:
        avg = round(float(np.mean(level_scores[tp])), 3)
        print(f"  tp={tp:.1f}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg > 0.0,
            expectation=f"Shifting short tp={tp} (transforms): accuracy={avg:.3f}"
        )
    overall = round(float(np.mean(all_scenario_avgs)), 3)
    print(f"\nScore: {overall:.3f} | {len(all_scenario_avgs)} scenarios | {time.time()-_start:.1f}s")
    return overall

zelusbench_shifting_short.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_shifting_short